In [1]:
import pandas as pd
import numpy as np
import random
import re

In [3]:
production = pd.read_csv("production_event.csv")
workorder = pd.read_csv("work_orders(1).csv")
quality = pd.read_csv("quality_inspection(1).csv")
sensor = pd.read_csv("sensor_readings(2).csv")
tool_master = pd.read_csv("tool_master(3).csv")
bom = pd.read_csv("bom_master(3).csv")

In [3]:
production["Timestamp"] = pd.to_datetime(
    production["Timestamp"],
    errors="coerce"
)

C:\Users\soumy\AppData\Local\Temp\ipykernel_1236\1476187888.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  production["Timestamp"] = pd.to_datetime(


In [5]:
# Clean Station column
production["Station"] = (
    production["Station"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [6]:
# Calculate Cycle Time (in seconds)
production["Cycle_Time_Sec"] = (
    production.groupby("Station")["Timestamp"]
    .diff()
    .dt.total_seconds()
)


In [7]:
# Check Cycle Time Plausibility
def cycle_status(time):
    if time == 0:
        return "First Record"
    elif time < 30:
        return "Too Fast"
    elif time > 300:
        return "Too Slow"
    else:
        return "Normal"

production["Cycle_Time_Status"] = production["Cycle_Time_Sec"].apply(cycle_status)

In [8]:
# Display result
print(production[
    [
        "Station",
        "Timestamp",
        "Cycle_Time_Sec",
        "Cycle_Time_Status"
    ]
].head(20))

   Station           Timestamp  Cycle_Time_Sec Cycle_Time_Status
0   STN-01                 NaT             NaN            Normal
1   STN-05 2026-03-01 06:01:00             NaN            Normal
2   STN-01 2026-03-01 06:02:00             NaN            Normal
3   STN-05 2026-03-01 06:03:00           120.0            Normal
4   STN-03 2026-03-01 06:05:00             NaN            Normal
5   STN-02 2026-03-01 06:06:00             NaN            Normal
6   STN-03 2026-03-01 06:07:00           120.0            Normal
7   STN-04 2026-03-01 06:08:00             NaN            Normal
8   STN-05 2026-03-01 06:10:00           420.0          Too Slow
9   STN-03 2026-03-01 06:11:00           240.0            Normal
10  STN-03 2026-03-01 06:12:00            60.0            Normal
11  STN-05 2026-03-01 06:13:00           180.0            Normal
12  STN-05 2026-03-01 06:15:00           120.0            Normal
13  STN-03 2026-03-01 06:16:00           240.0            Normal
14  STN-04 2026-03-01 06:

In [ ]:
# Save cleaned dataset

production.to_csv(
    "production_events_cleaned.csv",
    index=False
)

print("\nCycle Time Plausibility Check Completed Successfully!")


Cycle Time Plausibility Check Completed Successfully!


In [ ]:
# tast case-12

In [13]:
# Remove extra spaces
production["Work_Order"] = production["Work_Order"].astype(str).str.strip().str.upper()
production["VIN"] = production["VIN"].astype(str).str.strip().str.upper()

workorder["Work_Order"] = workorder["Work_Order"].astype(str).str.strip().str.upper()
workorder["VIN"] = workorder["VIN"].astype(str).str.strip().str.upper()


In [14]:
# Merge Production with Work Order Master
production = production.merge(
    workorder[["Work_Order", "VIN"]],
    on="Work_Order",
    how="left",
    suffixes=("", "_Master")
)


In [15]:
# Validate VIN
production["WO_VIN_Status"] = np.where(
    production["VIN"] == production["VIN_Master"],
    "Valid",
    "Invalid"
)

In [16]:
# Display Result
print(production[
    [
        "Work_Order",
        "VIN",
        "VIN_Master",
        "WO_VIN_Status"
    ]
].head(20))


   Work_Order                VIN         VIN_Master WO_VIN_Status
0     WO10000          XX1 2 3 4  MA1AB100000000000       Invalid
1     WO10001  MA1AB100000000001  MA1AB100000000001         Valid
2     WO10002  MA1AB100000000002  MA1AB100000000002         Valid
3     WO10003  MA1AB100000000003  MA1AB100000000003         Valid
4     WO10004  MA1AB100000000004  MA1AB100000000004         Valid
5     WO10005  MA1AB100000000005  MA1AB100000000005         Valid
6     WO10006  MA1AB100000000006  MA1AB100000000006         Valid
7     WO10007  MA1AB100000000007  MA1AB100000000007         Valid
8     WO10008  MA1AB100000000008  MA1AB100000000008         Valid
9     WO10009  MA1AB100000000009  MA1AB100000000009         Valid
10    WO10010  MA1AB100000000010  MA1AB100000000010         Valid
11    WO10011  MA1AB100000000011  MA1AB100000000011         Valid
12    WO10012  MA1AB100000000012  MA1AB100000000012         Valid
13    WO10013  MA1AB100000000013  MA1AB100000000013         Valid
14    WO10

In [17]:
# Save cleaned dataset
production.to_csv(
    "production_events_cleaned.csv",
    index=False
)

print("\nWork Order Validation Completed Successfully!")


Work Order Validation Completed Successfully!


In [ ]:
# test case = 13

In [18]:
# Convert Timestamp to datetime
production["Timestamp"] = pd.to_datetime(
    production["Timestamp"],
    errors="coerce"
)


C:\Users\soumy\AppData\Local\Temp\ipykernel_31304\4069202784.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  production["Timestamp"] = pd.to_datetime(


In [19]:
# Clean columns
production["VIN"] = (
    production["VIN"]
    .astype(str)
    .str.strip()
    .str.upper()
)

production["Station"] = (
    production["Station"]
    .astype(str)
    .str.strip()
    .str.upper()
)


In [20]:
# Sort data
production = production.sort_values(
    by=["VIN", "Station", "Timestamp"]
)

# Count station visits
production["Visit_No"] = (
    production.groupby(["VIN", "Station"])
    .cumcount() + 1
)


In [21]:
# Tag Rework
production["Rework_Status"] = production["Visit_No"].apply(
    lambda x: "Rework" if x > 1 else "Normal"
)


In [22]:
# Display result
print(production[
    [
        "VIN",
        "Station",
        "Timestamp",
        "Visit_No",
        "Rework_Status"
    ]
].head(20))


                    VIN Station           Timestamp  Visit_No Rework_Status
1     MA1AB100000000001  STN-05 2026-03-01 06:01:00         1        Normal
5001  MA1AB100000000001  STN-05 2026-03-01 06:01:00         2        Rework
2     MA1AB100000000002  STN-01 2026-03-01 06:02:00         1        Normal
5002  MA1AB100000000002  STN-01 2026-03-01 06:02:00         2        Rework
3     MA1AB100000000003  STN-05 2026-03-01 06:03:00         1        Normal
5003  MA1AB100000000003  STN-05 2026-03-01 06:03:00         2        Rework
4     MA1AB100000000004  STN-03 2026-03-01 06:05:00         1        Normal
5004  MA1AB100000000004  STN-03 2026-03-01 06:05:00         2        Rework
5     MA1AB100000000005  STN-02 2026-03-01 06:06:00         1        Normal
5005  MA1AB100000000005  STN-02 2026-03-01 06:06:00         2        Rework
6     MA1AB100000000006  STN-03 2026-03-01 06:07:00         1        Normal
5006  MA1AB100000000006  STN-03 2026-03-01 06:07:00         2        Rework
7     MA1AB1

In [23]:
# Save cleaned dataset
production.to_csv(
    "production_events_cleaned.csv",
    index=False
)

print("\nRework Loop Identification Completed Successfully!")


Rework Loop Identification Completed Successfully!


In [ ]:
# test case = 14

In [24]:
# Clean existing Shift column
production["Shift"] = (
    production["Shift"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [25]:
# Function to derive shift from timestamp
def derive_shift(timestamp):
    if pd.isna(timestamp):
        return "UNKNOWN"

    hour = timestamp.hour

    if 6 <= hour < 14:
        return "A"
    elif 14 <= hour < 22:
        return "B"
    else:
        return "C"


In [26]:
# Create derived shift
production["Derived_Shift"] = production["Timestamp"].apply(derive_shift)

In [27]:
# Validate shift
production["Shift_Status"] = np.where(
    production["Shift"] == production["Derived_Shift"],
    "Valid",
    "Invalid"
)

In [28]:
# Display result
print(production[
    [
        "Timestamp",
        "Shift",
        "Derived_Shift",
        "Shift_Status"
    ]
].head(20))

               Timestamp Shift Derived_Shift Shift_Status
1    2026-03-01 06:01:00   NAN             A      Invalid
5001 2026-03-01 06:01:00   NAN             A      Invalid
2    2026-03-01 06:02:00     A             A        Valid
5002 2026-03-01 06:02:00     A             A        Valid
3    2026-03-01 06:03:00     B             A      Invalid
5003 2026-03-01 06:03:00     B             A      Invalid
4    2026-03-01 06:05:00   NAN             A      Invalid
5004 2026-03-01 06:05:00   NAN             A      Invalid
5    2026-03-01 06:06:00     A             A        Valid
5005 2026-03-01 06:06:00     A             A        Valid
6    2026-03-01 06:07:00   NAN             A      Invalid
5006 2026-03-01 06:07:00   NAN             A      Invalid
7    2026-03-01 06:08:00     C             A      Invalid
5007 2026-03-01 06:08:00     C             A      Invalid
8    2026-03-01 06:10:00     C             A      Invalid
5008 2026-03-01 06:10:00     C             A      Invalid
9    2026-03-0

In [29]:
# Save cleaned dataset
production.to_csv(
    "production_events_cleaned.csv",
    index=False
)

print("\nShift Code Derivation and Validation Completed Successfully!")


Shift Code Derivation and Validation Completed Successfully!


In [ ]:
# test case = 15

In [4]:
print(quality.columns.tolist())

['Inspection_ID', 'Event_ID', 'VIN', 'Defect_Code', 'Defect', 'Inspection_Source', 'Inspector_ID', 'Rework_Flag', 'Scrap_Reason', 'Result']


In [10]:

scrap_codes = ["SR01","SR02","SR03","SR04","SR05","SR06"]

quality["Scrap_Code"] = [
    random.choice(scrap_codes) if x=="REJECT" else None
    for x in quality["Defect"]
]

In [11]:
scrap_mapping = {
    "SR01":"Welding Defect",
    "SR02":"Paint Defect",
    "SR03":"Crack",
    "SR04":"Dimension Error",
    "SR05":"Material Defect",
    "SR06":"Electrical Failure"
}

quality["Scrap_Reason"] = quality["Scrap_Code"].map(scrap_mapping)

In [8]:
# View original scrap codes
print("Before Mapping:")
print(quality["Scrap_Code"].head(10))

Before Mapping:
0    None
1    None
2    None
3    None
4    None
5    None
6    None
7    None
8    None
9    None
Name: Scrap_Code, dtype: object


In [ ]:
#test case = 16

In [15]:
# Torque Conversion to Nm
# -----------------------------
def convert_torque(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()

    number = re.findall(r"\d+\.?\d*", value)

    if len(number) == 0:
        return np.nan

    number = float(number[0])

    if "LB-FT" in value or "LBFT" in value:
        return round(number * 1.35582,2)

    else:
        return number

In [16]:
# -----------------------------
# Temperature Conversion to Celsius
# -----------------------------
def convert_temperature(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()

    number = re.findall(r"\d+\.?\d*", value)

    if len(number) == 0:
        return np.nan

    number = float(number[0])

    if "F" in value:
        return round((number-32)*5/9,2)

    else:
        return number


In [17]:
# -----------------------------
# Pressure Conversion to bar
# -----------------------------
def convert_pressure(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()

    number = re.findall(r"\d+\.?\d*", value)

    if len(number) == 0:
        return np.nan

    number = float(number[0])

    if "PSI" in value:
        return round(number*0.0689476,2)

    else:
        return number

In [18]:
# Apply conversions
sensor["Torque_Nm"] = sensor["Torque"].apply(convert_torque)

sensor["Temperature_C"] = sensor["Temperature"].apply(convert_temperature)

sensor["Pressure_bar"] = sensor["Pressure"].apply(convert_pressure)

In [19]:
# Display output
print(sensor[
[
"Torque",
"Torque_Nm",
"Temperature",
"Temperature_C",
"Pressure",
"Pressure_bar"
]
].head(20))

      Torque  Torque_Nm Temperature  Temperature_C Pressure  Pressure_bar
0        NaN        NaN         85C           85.0  4.5 bar          4.50
1      45 Nm      45.00        35 C           35.0  5.2 bar          5.20
2       60Nm      60.00         85C           85.0   65 psi          4.48
3        NaN        NaN        35 C           35.0  4.5 bar          4.50
4   35 lb-ft      47.45       104 F           40.0  4.5 bar          4.50
5      50 Nm      50.00        35 C           35.0  4.5 bar          4.50
6   72 lb-ft      97.62         85C           85.0   65 psi          4.48
7      50 Nm      50.00       104 F           40.0   72 psi          4.96
8      50 Nm      50.00        35 C           35.0  5.2 bar          5.20
9   35 lb-ft      47.45        95 F           35.0  5.2 bar          5.20
10  72 lb-ft      97.62        35 C           35.0   72 psi          4.96
11  72 lb-ft      97.62         85C           85.0  4.5 bar          4.50
12     45 Nm      45.00       104 F   

In [20]:
# Save cleaned data
sensor.to_csv("sensor_readings_cleaned.csv",index=False)

print("\nUnit Conversion Completed Successfully!")


Unit Conversion Completed Successfully!


In [ ]:
# test case =17

In [21]:
# Convert columns to numeric
sensor["Torque"] = pd.to_numeric(sensor["Torque"], errors="coerce")
sensor["Temperature"] = pd.to_numeric(sensor["Temperature"], errors="coerce")
sensor["Pressure"] = pd.to_numeric(sensor["Pressure"], errors="coerce")

In [22]:
# Function to detect anomalies
def detect_anomaly(row):

    if (
        row["Torque"] < 40 or row["Torque"] > 60 or
        row["Temperature"] < 70 or row["Temperature"] > 100 or
        row["Pressure"] < 4 or row["Pressure"] > 8
    ):
        return "Out of Spec"

    return "Normal"


In [23]:
# Create anomaly status
sensor["Anomaly_Status"] = sensor.apply(
    detect_anomaly,
    axis=1
)

In [24]:
# Show results
print(sensor[
[
    "Sensor_ID",
    "Torque",
    "Temperature",
    "Pressure",
    "Anomaly_Status"
]
].head(20))


   Sensor_ID  Torque  Temperature  Pressure Anomaly_Status
0    SEN-034     NaN          NaN       NaN         Normal
1    SEN-037     NaN          NaN       NaN         Normal
2    SEN-038     NaN          NaN       NaN         Normal
3    SEN-008     NaN          NaN       NaN         Normal
4    SEN-003     NaN          NaN       NaN         Normal
5    SEN-048     NaN          NaN       NaN         Normal
6    SEN-048     NaN          NaN       NaN         Normal
7    SEN-010     NaN          NaN       NaN         Normal
8    SEN-018     NaN          NaN       NaN         Normal
9    SEN-035     NaN          NaN       NaN         Normal
10   SEN-039     NaN          NaN       NaN         Normal
11   SEN-012     NaN          NaN       NaN         Normal
12   SEN-005     NaN          NaN       NaN         Normal
13   SEN-018     NaN          NaN       NaN         Normal
14   SEN-035     NaN          NaN       NaN         Normal
15   SEN-005     NaN          NaN       NaN         Norm

In [25]:
# Count anomalies
print("\nSummary:")
print(sensor["Anomaly_Status"].value_counts())


Summary:
Anomaly_Status
Normal    10000
Name: count, dtype: int64


In [26]:
# Save cleaned dataset
sensor.to_csv(
    "sensor_readings_cleaned.csv",
    index=False
)

print("\nAnomaly Detection Completed Successfully!")


Anomaly Detection Completed Successfully!


In [ ]:
# test case = 18

In [27]:
# Remove extra spaces
quality["Inspector_ID"] = (
    quality["Inspector_ID"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [28]:
# Function to identify inspection source
def inspection_source(inspector):

    if inspector.startswith("EMP"):
        return "Human"

    elif inspector.startswith("CAM"):
        return "Automated"

    elif inspector.startswith("ROB"):
        return "Automated"

    elif inspector.startswith("AUTO"):
        return "Automated"

    else:
        return "Unknown"

In [29]:
# Create new column
quality["Inspection_Source"] = quality["Inspector_ID"].apply(
    inspection_source
)

In [30]:
# Display result
print(quality[
[
    "Inspector_ID",
    "Inspection_Source"
]].head(20))

   Inspector_ID Inspection_Source
0         INS69           Unknown
1         INS83           Unknown
2         INS58           Unknown
3         INS83           Unknown
4         INS31           Unknown
5         INS22           Unknown
6         INS50           Unknown
7         INS48           Unknown
8         INS55           Unknown
9         INS66           Unknown
10        INS53           Unknown
11        INS14           Unknown
12        INS51           Unknown
13        INS95           Unknown
14        INS38           Unknown
15        INS84           Unknown
16        INS10           Unknown
17        INS23           Unknown
18        INS79           Unknown
19        INS99           Unknown


In [31]:
# Summary
print("\nInspection Source Count:")
print(quality["Inspection_Source"].value_counts())


Inspection Source Count:
Inspection_Source
Unknown    4000
Name: count, dtype: int64


In [33]:
# Save dataset
quality.to_csv(
    "quality_inspection_cleaned.csv",
    index=False
)

print("\nHuman vs Automated Inspection Tagging Completed Successfully!")


Human vs Automated Inspection Tagging Completed Successfully!


In [ ]:
#test case = 19

In [36]:
# Clean Tool_ID
production["Tool_ID"] = (
    production["Tool_ID"]
    .astype(str)
    .str.strip()
    .str.upper()
)

tool_master["Tool_ID"] = (
    tool_master["Tool_ID"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [7]:

# Merge production with tool master
production = pd.merge(
    production,
    tool_master,
    on="Tool_ID",
    how="left"
)


In [8]:
# Replace missing values
production["Tool_Name"] = production["Tool_Name"].fillna("Unknown Tool")
production["Tool_Type"] = production["Tool_Type"].fillna("Unknown Type")

In [9]:
# Display result
print(production[
    ["Tool_ID", "Tool_Name", "Tool_Type"]
].head(10))

  Tool_ID Tool_Name    Tool_Type
0  TL-032   Tool_32   Press Tool
1  TL-076   Tool_76  Welding Gun
2  TL-078   Tool_78   Press Tool
3  TL-054   Tool_54   Press Tool
4  TL-090   Tool_90   Nut Runner
5  TL-014   Tool_14   Nut Runner
6  TL-006    Tool_6   Nut Runner
7  TL-081   Tool_81   Nut Runner
8  TL-099   Tool_99  Welding Gun
9  TL-059   Tool_59    Robot Arm


In [10]:
# Save output
production.to_csv("production_events_cleaned.csv", index=False)

print("Tool ID Mapping Completed Successfully!")

Tool ID Mapping Completed Successfully!


In [ ]:
#test case = 20

In [5]:
print(bom.columns.tolist())

['Part_Number', 'Part_Name', 'Category', 'BOM_Version', 'Effective_Date', 'Unit_Cost_USD', 'Supplier_Code', 'Status', 'Plant']


In [ ]:
bom.rename(columns={
    "Start Date": "Start_Date",
    "End Date": "End_Date"
}, inplace=True)

In [9]:
bom["Start Date"] = pd.to_datetime(bom["Start Date"])
bom["End Date"] = pd.to_datetime(bom["End Date"])

KeyError: 'Start Date'